In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
texte = [
    "freight forwarding and logistics company",
    "company that transports goods for other businesses",
    "manufacturer of forklifts for warehouses",
    "pharmaceutical company developing new medicines",
    "software for managing logistics",
]
vectori = model.encode(texte)
vectori.shape

(5, 384)

In [3]:
from sentence_transformers import util
util.cos_sim(vectori, vectori)

tensor([[1.0000, 0.6839, 0.4544, 0.2289, 0.5535],
        [0.6839, 1.0000, 0.5527, 0.2691, 0.4984],
        [0.4544, 0.5527, 1.0000, 0.2195, 0.2926],
        [0.2289, 0.2691, 0.2195, 1.0000, 0.2082],
        [0.5535, 0.4984, 0.2926, 0.2082, 1.0000]])

cele 2 numere din vectori.shape reprezinta (numarul de linii = 5 si numarul de numere din vectorul fiecarui text = 384)

textul 1 si 2 desi nu au nici un cuvand in comun, modelul le vede destul de asemanatoare(se repeta 1, 0.6839)

textul 3 si textul 5 sunt si ele relativ apropiate de textul 1, desi nu ar trebui sa fie asa (textul 1 = companie de logistica, textul 3 = face stivuitoare, textul 5 = face software de logistica)

observam ca fata de textul 1, textul 2 este singurul asemanator cu el, in rest ce celelalte nu au treaba, asta inseamna ca, o sa avem nevoie si de un LLM pentru cazurile apropiate (textul 2 = 0,68 da, umatorul textul 5 = 0,55 nu)

In [5]:
import pandas as pd
df = pd.read_json("companies.jsonl", lines=True)

In [6]:
import ast

# 1. definesc functia (intai!)
def to_dict(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

# 2. o aplic pe adresa (dupa ce e definita)
df["address"] = df.address.apply(to_dict)

# 3. scot tara
df["country"] = df.address.str.get("country_code")

# 4. firmele romanesti
ro = df[df.country == "ro"].copy()
len(ro)

26

In [7]:
v_firme = model.encode(ro.description.tolist())
v_cerere = model.encode("Logistic companies in Romania")
ro["scor"] = util.cos_sim(v_cerere, v_firme)[0].numpy()

In [12]:
ro.sort_values(by="scor", ascending=False)[["operational_name", "scor"]]

,operational_name,scor
11,Bunge Romania,0.528072
3,CBRE Romania,0.489444
13,METRO România,0.466078
8,Dacia,0.458969
20,Brasov Industrial Portfolio,0.441473
23,European Drinks,0.438837
21,STILL,0.431440
7,Mercedes-Benz Trucks & Buses Romania,0.430960
15,Poșta Română,0.398202
6,Compania Națională de Căi Ferate CFR,0.391626


cele 6 firme (CFR, Portul Constanta, Posta, Brasov Industrial Portfolio, Romgaz, Valbur) nu apar in partea de sus a tabelului, din ele corecta e doar 1 din 6 (Brasov Industrial Portfolio)

Posta pe 9, CFR pe 10, Valbur pe 12, Portul Constanta pe 18, Romgaz pe 19. Portul e practic printre ultimele

Rompetrol apare de două ori (locurile 17 si 24), din cauza dublurii

STILL este pe locul 7 si Rompetrol mai jos pe locul 17 

daca am lua primele 6 din lista, acestea ar fi (Bunge Romania, CBRE Romania, METRO România, Dacia, Brasov Industrial Portfolio, European Drinks)

In [17]:
v_cerere = model.encode("logistics company providing transport, freight and warehousing services for other businesses")
ro["scor"] = util.cos_sim(v_cerere, v_firme)[0].numpy()
ro.sort_values(by="scor", ascending=False)[["operational_name", "scor"]]

,operational_name,scor
20,Brasov Industrial Portfolio,0.457185
6,Compania Națională de Căi Ferate CFR,0.392100
9,Fildas Trading,0.391482
13,METRO România,0.390247
21,STILL,0.381184
7,Mercedes-Benz Trucks & Buses Romania,0.380496
16,OSCAR,0.359240
18,Valtec,0.297720
65,Sincron HR,0.295116
10,Portul Constanta,0.294331


observam ca dupa ce am schimbat cererea, firmele s au reordonat

din cele 6 initial cu da (CFR, Portul Constanta, Posta, Brasov Industrial Portfolio, Romgaz, Valbur) se regasesc acum 2 (Brasov Industrial Portfolio pe 1 si CRF pe 2)

CFR a ajuns de pe pozitia 10 pe 2, Portul de pe 18 pe 10 si STILL de pe 7 pe 5

observam ca, embeddings sunt utile ca sa ordoneze aproximativ, dar nu pot face distinctia de rol. pentru asta e nevoie de LLM.